[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/078.%20The%20Fleet%20Sizing%20and%20Composition%20Problem/P78-Tier-8_executed.ipynb)

---


# 78. The Fleet Sizing and Composition Problem
## Tier 8: Value-Aligned and Ethical Fleet Evaluation

### Purpose and scope

This notebook evaluates a small fleet-allocation problem using a value-aligned framework. It represents vehicle types, routes, stakeholder groups, ethical principles, and four normalized objectives: economic efficiency, environmental sustainability, social service, and safety. A finite set of candidate allocations is scored, and the highest-scoring feasible candidate is selected.

The method is an illustrative weighted-sum evaluation over a generated candidate set; it is not a proof of a global optimum over every possible fleet. The numerical scores are modeling proxies, not validated measures of ethics or social welfare. The framework does not establish that a decision is ethically acceptable merely because it receives a high score.

### Modeling assumptions

- Each allocation entry is a nonnegative integer count of vehicles of one type assigned to one route.
- The route distance used for emissions scoring is assumed to be 100 km per vehicle. Acquisition and route operating costs are added once per assigned vehicle for the modeled planning period.
- Route demand is measured in cargo units. Service is capped at 100% when assigned capacity exceeds demand.
- Budget, per-vehicle emissions, noise, minimum service equity, and minimum average safety are treated as feasibility checks. The cost-increase threshold is reported through the economic compliance score rather than used as a hard feasibility check.
- Stakeholder impacts, equity, sustainability, and transparency are simplified indicators. They omit many real-world effects, such as changing traffic conditions, lifecycle emissions, labor contracts, geographic distribution within routes, and stakeholder consultation.
- Candidate variations are stochastic. The explicitly listed base allocations are included in every run; the candidate set is not exhaustive, and no random seed is specified.

### Method overview

1. Define the fleet, routes, stakeholder groups, principles, objectives, and thresholds.
2. Construct a finite candidate set from five representative allocations and small random variations.
3. Reject candidates that fail the stated hard feasibility checks.
4. Score each remaining candidate using objective, compliance, equity, sustainability, and transparency measures.
5. Select the candidate with the highest weighted ethical score and report its metrics and reasoning.

The framework uses a weighted sum rather than a Pareto-frontier method. For candidate allocation \(x\), its selection score is

$$
S(x) = 0.3 O(x) + 0.3 C(x) + 0.2 E(x) + 0.1 U(x) + 0.1 T(x),
$$

where \(O\) is the weighted mean of the four normalized objective scores, \(C\) is the principle-weighted mean of compliance scores, \(E\) is stakeholder equity, \(U\) is sustainability, and \(T\) is transparency. All five components are intended to lie in \([0,1]\). The selected allocation maximizes \(S(x)\) over the feasible candidates generated in that run.

This distinction matters: feasibility checks are pass-or-fail rules, while the weighted score ranks the candidates that pass them. The weights encode modeling choices and can affect the ranking; they should not be interpreted as universally correct ethical priorities.

### Mathematical representation

Let \(i \in I\) index vehicle types and \(j \in J\) index routes. The decision \(x_{ij} \in \mathbb{Z}_{\geq 0}\) is the number of vehicles of type \(i\) assigned to route \(j\). The implementation stores these decisions in a vehicles-by-routes NumPy array named `fleet_allocation`.

For route \(j\), assigned capacity and capped service level are

$$
K_j(x) = \sum_{i \in I} q_i x_{ij},
\qquad
s_j(x) = \min\left(1, \frac{K_j(x)}{d_j}\right),
$$

where \(q_i\) is vehicle capacity and \(d_j\) is route demand. If demand is zero, the implementation defines the service level as 1. The social objective is the minimum route service level, \(\min_{j \in J} s_j(x)\), so a well-served route cannot conceal a poorly served one.

The modeled total cost and emissions are

$$
C_{\mathrm{total}}(x) = \sum_{i \in I}\sum_{j \in J} x_{ij}(a_i + c_{ij}),
\qquad
G(x) = 100\sum_{i \in I}\sum_{j \in J} x_{ij} g_i,
$$

where \(a_i\) is acquisition cost, \(c_{ij}\) is route operating cost, and \(g_i\) is emissions in kg CO\(_2\) per km. The factor 100 is the assumed route distance in km. These equations are implemented by `_calculate_total_cost`, `_calculate_total_emissions`, and `_calculate_objectives`.

A candidate is feasible only if it meets the budget limit, the per-vehicle emissions and noise thresholds, the minimum route service-equity threshold, and the minimum average safety threshold. The finite-candidate optimization is therefore

$$
\max_{x \in \mathcal{F}} S(x),
$$

where \(\mathcal{F}\) is the subset of generated candidates that pass those checks. This is not an exhaustive integer-programming search: its result is the best-scoring feasible candidate in the generated pool. If several candidates have scores equal within a numerical tolerance, the implementation keeps the first one encountered.

The candidate-generation step creates five specified allocations and three random, rounded, nonnegative variations of each. If \(m\) candidates are generated and each allocation has \(|I||J|\) entries, evaluating them takes \(O(m|I||J|)\) time, apart from the small stakeholder and principle calculations. The finite pool makes the demonstration easy to inspect, but does not guarantee a global optimum.

### Interpreting the results

A high score means that a candidate ranks well under the stated proxy measures and weights among the feasible candidates generated in that run. It does not establish that every stakeholder is treated fairly, that every ethical principle is satisfied in a broader sense, or that the allocation is globally optimal. Inspect the objective scores, compliance scores, feasibility checks, and stakeholder indicators separately rather than relying on one aggregate number.

The framework is useful here as a structured teaching example: it makes value choices explicit, distinguishes hard feasibility checks from soft scores, and produces a record of the indicators used to rank a decision.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
import time

import numpy as np


@dataclass
class VehicleType:
    """Vehicle characteristics used by the fleet-allocation model."""

    id: int
    name: str
    capacity: int
    acquisition_cost: float
    operating_costs: Dict[int, float]
    service_rate: float
    carbon_emissions: float
    noise_level: float
    safety_rating: float
    accessibility: float


@dataclass
class Route:
    """Route demand and contextual characteristics."""

    id: int
    name: str
    demand_units: int
    arrival_rate: float
    max_delay_prob: float = 0.1
    population_density: float = 1.0
    environmental_sensitivity: float = 1.0
    social_equity_factor: float = 1.0


@dataclass
class Stakeholder:
    """Stakeholder group and its modeled priorities."""

    id: int
    name: str
    stakeholder_type: str
    priorities: Dict[str, float]
    influence_level: float
    vulnerability: float
    equity_concern: float


@dataclass
class EthicalPrinciple:
    """Principle metadata used to weight compliance scores."""

    id: int
    name: str
    description: str
    constraints: List[str]
    weight: float
    violation_penalty: float
    measurable: bool


@dataclass
class EthicalDecision:
    """Candidate allocation together with its evaluated indicators."""

    fleet_allocation: np.ndarray
    objectives: Dict[str, float]
    ethical_compliance: Dict[str, float]
    stakeholder_impacts: Dict[int, Dict[str, float]]
    equity_score: float
    sustainability_score: float
    transparency_score: float
    total_ethical_score: float
    reasoning: List[str]


@dataclass
class EthicalResult:
    """Report data for the selected candidate."""

    optimal_fleet: Dict[Tuple[int, int], int]
    total_cost: float
    multi_objective_scores: Dict[str, float]
    ethical_compliance: Dict[str, float]
    stakeholder_equity: Dict[int, float]
    sustainability_metrics: Dict[str, float]
    transparency_metrics: Dict[str, float]
    computation_time: float
    insights: List[str]

## Define the value framework

The framework makes its modeling choices explicit before evaluating any allocation. Principle weights and objective weights describe relative importance within this example; they are not learned from stakeholder data. The stakeholder profiles provide vulnerability and priority information for the simplified impact indicators.

In [ ]:
class ConstitutionalAIFramework:
    """Store principles, stakeholders, objectives, and evaluation rules."""

    def __init__(self, vehicles: List[VehicleType], routes: List[Route]):
        self.vehicles = vehicles
        self.routes = routes
        self.num_vehicles = len(vehicles)
        self.num_routes = len(routes)
        self.ethical_principles = self._initialize_ethical_principles()
        self.stakeholders = self._initialize_stakeholders()
        self.objectives = self._initialize_objectives()
        self.ethical_constraints = self._define_ethical_constraints()
        self.equity_weights = self._define_equity_weights()
        self.sustainability_targets = self._define_sustainability_targets()

        print("Constitutional framework initialized:")
        print(f"  Ethical principles: {len(self.ethical_principles)}")
        print(f"  Stakeholder groups: {len(self.stakeholders)}")
        print(f"  Objectives: {len(self.objectives)}")
        print(f"  Feasibility and evaluation thresholds: {len(self.ethical_constraints)}")

    def _initialize_ethical_principles(self) -> List[EthicalPrinciple]:
        """Define the principles and their contribution to compliance scoring."""
        return [
            EthicalPrinciple(
                1, "Environmental Responsibility",
                "Limit emissions and noise while considering environmental protection.",
                ["Per-vehicle emissions threshold", "Noise threshold"],
                0.25, 0.8, True
            ),
            EthicalPrinciple(
                2, "Social Equity",
                "Provide adequate service across the modeled routes.",
                ["Minimum service level across routes"],
                0.25, 0.9, True
            ),
            EthicalPrinciple(
                3, "Safety and Well-being",
                "Maintain an acceptable average vehicle safety rating.",
                ["Minimum average safety score"],
                0.20, 0.95, True
            ),
            EthicalPrinciple(
                4, "Economic Viability",
                "Keep the modeled cost within a reference range.",
                ["Cost increase relative to the reference cost"],
                0.15, 0.6, True
            ),
            EthicalPrinciple(
                5, "Transparency and Accountability",
                "Record indicators and provide a concise explanation of the selection.",
                ["Reported metrics and decision reasoning"],
                0.15, 0.5, True
            ),
        ]

    def _initialize_stakeholders(self) -> List[Stakeholder]:
        """Specify the groups represented by the simplified impact model."""
        return [
            Stakeholder(
                1, "Customers", "customers",
                {"service_quality": 0.4, "cost": 0.3, "reliability": 0.2, "accessibility": 0.1},
                0.3, 0.6, 0.7
            ),
            Stakeholder(
                2, "Employees", "employees",
                {"safety": 0.4, "wages": 0.3, "working_conditions": 0.2, "job_security": 0.1},
                0.2, 0.4, 0.6
            ),
            Stakeholder(
                3, "Local Communities", "community",
                {"environmental_quality": 0.3, "noise_reduction": 0.2,
                 "local_jobs": 0.2, "community_benefits": 0.3},
                0.2, 0.8, 0.9
            ),
            Stakeholder(
                4, "Environment", "environment",
                {"carbon_reduction": 0.4, "biodiversity": 0.3,
                 "pollution_control": 0.2, "sustainability": 0.1},
                0.1, 0.9, 1.0
            ),
            Stakeholder(
                5, "Investors", "investors",
                {"profitability": 0.5, "risk_management": 0.3,
                 "growth": 0.1, "sustainability": 0.1},
                0.2, 0.2, 0.3
            ),
        ]

    def _initialize_objectives(self) -> Dict[str, Dict[str, Any]]:
        """Define normalized objectives and their weighted-sum weights."""
        return {
            "economic": {
                "name": "Economic Efficiency", "direction": "maximize", "weight": 0.30
            },
            "environmental": {
                "name": "Environmental Sustainability", "direction": "maximize", "weight": 0.25
            },
            "social": {
                "name": "Social Responsibility", "direction": "maximize", "weight": 0.25
            },
            "safety": {
                "name": "Safety and Well-being", "direction": "maximize", "weight": 0.20
            },
        }

    def _define_ethical_constraints(self) -> Dict[str, float]:
        """Set the numerical thresholds used by feasibility and compliance checks."""
        return {
            "max_carbon_per_vehicle": 50.0,
            "max_noise_level": 70.0,
            "min_service_equity": 0.8,
            "max_cost_increase": 0.2,
            "min_safety_score": 0.8,
            "max_vulnerability_impact": 0.3,
            "budget_limit": 500000.0,
            "baseline_cost": 320000.0,
        }

    def _define_equity_weights(self) -> Dict[str, float]:
        """Set factors used in the stakeholder equity indicator."""
        return {
            "vulnerability_multiplier": 2.0,
            "equity_threshold": 0.8,
            "distribution_fairness": 0.7,
            "accessibility_requirement": 0.9,
        }

    def _define_sustainability_targets(self) -> Dict[str, float]:
        """Record contextual targets not directly optimized in this example."""
        return {
            "carbon_reduction_target": 0.3,
            "renewable_energy_ratio": 0.5,
            "waste_reduction_target": 0.4,
            "biodiversity_protection": 0.8,
        }

    def _calculate_total_cost(self, fleet_allocation: np.ndarray) -> float:
        """Add acquisition and route operating costs for every assigned vehicle."""
        total_cost = 0.0
        for i, vehicle in enumerate(self.vehicles):
            for j, route in enumerate(self.routes):
                total_cost += fleet_allocation[i, j] * (
                    vehicle.acquisition_cost + vehicle.operating_costs[route.id]
                )
        return float(total_cost)

    def _calculate_total_emissions(self, fleet_allocation: np.ndarray) -> float:
        """Estimate emissions using the assumed 100 km route distance."""
        total_emissions = 0.0
        for i, vehicle in enumerate(self.vehicles):
            total_emissions += (
                np.sum(fleet_allocation[i, :]) * vehicle.carbon_emissions * 100
            )
        return float(total_emissions)

    def _service_levels(self, fleet_allocation: np.ndarray) -> List[float]:
        """Calculate capped demand coverage for each route."""
        service_levels = []
        for j, route in enumerate(self.routes):
            capacity = sum(
                fleet_allocation[i, j] * vehicle.capacity
                for i, vehicle in enumerate(self.vehicles)
            )
            service = min(1.0, capacity / route.demand_units) if route.demand_units > 0 else 1.0
            service_levels.append(float(service))
        return service_levels

    def _average_safety(self, fleet_allocation: np.ndarray) -> float:
        """Calculate vehicle-count-weighted average safety, including the empty-fleet case."""
        total_vehicles = float(np.sum(fleet_allocation))
        if total_vehicles == 0:
            return 1.0
        weighted_safety = sum(
            fleet_allocation[i, j] * vehicle.safety_rating
            for i, vehicle in enumerate(self.vehicles)
            for j in range(self.num_routes)
        )
        return float(weighted_safety / total_vehicles)

    def is_feasible(self, fleet_allocation: np.ndarray, context: Dict[str, Any]) -> bool:
        """Apply the stated hard checks before a candidate can be scored."""
        if fleet_allocation.shape != (self.num_vehicles, self.num_routes):
            return False
        if np.any(fleet_allocation < 0) or not np.allclose(fleet_allocation, np.rint(fleet_allocation)):
            return False
        if self._calculate_total_cost(fleet_allocation) > context["budget_limit"]:
            return False
        if min(self._service_levels(fleet_allocation)) < self.ethical_constraints["min_service_equity"]:
            return False
        if self._average_safety(fleet_allocation) < self.ethical_constraints["min_safety_score"]:
            return False

        max_carbon = self.ethical_constraints["max_carbon_per_vehicle"]
        max_noise = self.ethical_constraints["max_noise_level"]
        for i, vehicle in enumerate(self.vehicles):
            if np.sum(fleet_allocation[i, :]) > 0:
                if vehicle.carbon_emissions * 100 > max_carbon:
                    return False
                if vehicle.noise_level > max_noise:
                    return False
        return True

    def evaluate_ethical_compliance(self, fleet_allocation: np.ndarray) -> Dict[str, float]:
        """Return bounded proxy scores for the five modeled principles."""
        constraints = self.ethical_constraints
        selected = [
            vehicle
            for i, vehicle in enumerate(self.vehicles)
            if np.sum(fleet_allocation[i, :]) > 0
        ]

        if selected:
            carbon_ratios = [
                vehicle.carbon_emissions * 100 / constraints["max_carbon_per_vehicle"]
                for vehicle in selected
            ]
            noise_ratios = [
                vehicle.noise_level / constraints["max_noise_level"]
                for vehicle in selected
            ]
            environmental = max(0.0, 1.0 - max(max(carbon_ratios), max(noise_ratios)) + 1.0)
            environmental = min(1.0, environmental)
        else:
            environmental = 1.0

        service = min(self._service_levels(fleet_allocation))
        safety = self._average_safety(fleet_allocation)
        min_safety = constraints["min_safety_score"]
        safety_compliance = min(1.0, safety / min_safety) if safety < min_safety else safety

        total_cost = self._calculate_total_cost(fleet_allocation)
        baseline = constraints["baseline_cost"]
        max_cost = baseline * (1 + constraints["max_cost_increase"])
        economic = max(0.0, min(1.0, 1.0 - max(0.0, total_cost - baseline) / (max_cost - baseline)))

        # This indicator represents recorded, auditable metrics, not a measured social property.
        transparency = 0.8
        if np.sum(fleet_allocation) > 10:
            transparency *= 0.9
        if len(self.stakeholders) > 4:
            transparency *= 0.95

        return {
            "environmental": float(environmental),
            "social_equity": float(service),
            "safety": float(min(1.0, safety_compliance)),
            "economic": float(economic),
            "transparency": float(transparency),
        }

## Specify the fleet and routes

The data below define three vehicle types and two routes. Capacity and demand share the same cargo-unit measure. Acquisition cost and route operating cost are monetary amounts in the model's unspecified currency units; emissions are in kg CO\(_2\) per km, and noise is in decibels. The route distance used for the emissions calculation is a separate modeling assumption stated above.

In [ ]:
vehicles = [
    VehicleType(
        id=1,
        name="Small Truck",
        capacity=5,
        acquisition_cost=50000,
        operating_costs={1: 1000, 2: 1500},
        service_rate=4.0,
        carbon_emissions=0.15,
        noise_level=65.0,
        safety_rating=0.85,
        accessibility=0.9,
    ),
    VehicleType(
        id=2,
        name="Medium Truck",
        capacity=10,
        acquisition_cost=80000,
        operating_costs={1: 1200, 2: 1800},
        service_rate=3.5,
        carbon_emissions=0.25,
        noise_level=70.0,
        safety_rating=0.9,
        accessibility=0.8,
    ),
    VehicleType(
        id=3,
        name="Large Truck",
        capacity=20,
        acquisition_cost=120000,
        operating_costs={1: 1500, 2: 2000},
        service_rate=3.0,
        carbon_emissions=0.35,
        noise_level=75.0,
        safety_rating=0.95,
        accessibility=0.7,
    ),
]

routes = [
    Route(
        id=1,
        name="Urban Route",
        demand_units=15,
        arrival_rate=8.0,
        max_delay_prob=0.1,
        population_density=2.5,
        environmental_sensitivity=1.5,
        social_equity_factor=1.2,
    ),
    Route(
        id=2,
        name="Rural Route",
        demand_units=25,
        arrival_rate=4.0,
        max_delay_prob=0.1,
        population_density=0.5,
        environmental_sensitivity=0.8,
        social_equity_factor=1.0,
    ),
]

print("FLEET SIZING PROBLEM SETUP")
print("=" * 50)
print(f"Vehicle types: {len(vehicles)}")
for vehicle in vehicles:
    print(
        f"  {vehicle.name}: capacity {vehicle.capacity}, "
        f"emissions {vehicle.carbon_emissions:.2f} kg CO2/km, "
        f"noise {vehicle.noise_level:.1f} dB, "
        f"safety {vehicle.safety_rating:.0%}"
    )

print(f"\nRoutes: {len(routes)}")
for route in routes:
    print(
        f"  {route.name}: demand {route.demand_units} cargo units, "
        f"population density {route.population_density:.1f} people/km^2"
    )

## Initialize the framework

The five principle weights sum to 1, as do the four objective weights. Keeping these weights visible makes the scoring policy inspectable. The framework uses stakeholder vulnerability when calculating the equity indicator; influence and the detailed priority dictionaries are retained as profile information but are not used as decision weights in this simplified scoring rule.

In [ ]:
constitutional_ai = ConstitutionalAIFramework(vehicles, routes)

print("\nEthical principles:")
for principle in constitutional_ai.ethical_principles:
    print(f"  {principle.name}: weight {principle.weight:.2f}")

print("\nStakeholder groups:")
for stakeholder in constitutional_ai.stakeholders:
    print(
        f"  {stakeholder.name}: vulnerability {stakeholder.vulnerability:.1f}, "
        f"equity concern {stakeholder.equity_concern:.1f}"
    )

print("\nObjective weights:")
for key, objective in constitutional_ai.objectives.items():
    print(f"  {objective['name']}: {objective['weight']:.2f}")

assert np.isclose(sum(p.weight for p in constitutional_ai.ethical_principles), 1.0)
assert np.isclose(sum(o["weight"] for o in constitutional_ai.objectives.values()), 1.0)

## Generate, filter, and rank candidates

A candidate is a complete vehicles-by-routes allocation matrix. The listed allocations represent different emphases; random variations broaden the finite set without changing the method into an exhaustive search. Variations are rounded to integer vehicle counts because partial vehicles are not meaningful decisions. Feasibility is checked before scoring so that a high aggregate score cannot compensate for a violation of a hard threshold.

In [ ]:
class MultiObjectiveOptimizer:
    """Rank a finite candidate set by weighted ethical score."""

    def __init__(self, constitutional_ai: ConstitutionalAIFramework):
        self.constitutional_ai = constitutional_ai
        self.vehicles = constitutional_ai.vehicles
        self.routes = constitutional_ai.routes
        self.num_vehicles = len(self.vehicles)
        self.num_routes = len(self.routes)
        self.objective_weights = {
            key: objective["weight"]
            for key, objective in constitutional_ai.objectives.items()
        }
        self.constraint_weights = {
            principle.name: principle.weight
            for principle in constitutional_ai.ethical_principles
        }
        print("\nMulti-objective optimizer initialized")
        print(f"  Objectives: {len(self.objective_weights)}")
        print(f"  Ethical principles: {len(self.constraint_weights)}")
        print("  Ranking rule: weighted sum over feasible candidates")

    def optimize_fleet(self, context: Optional[Dict[str, Any]] = None) -> EthicalDecision:
        """Select the highest-scoring feasible candidate from the generated pool."""
        if context is None:
            context = self._create_default_context()

        candidates = self._generate_candidates(context)
        feasible_candidates = [
            candidate
            for candidate in candidates
            if self.constitutional_ai.is_feasible(candidate, context)
        ]
        if not feasible_candidates:
            raise ValueError("No generated candidate satisfies all hard feasibility checks.")

        evaluated = [
            self._evaluate_candidate(candidate, context)
            for candidate in feasible_candidates
        ]
        best_score = max(decision.total_ethical_score for decision in evaluated)
        tied_decisions = [
            decision
            for decision in evaluated
            if np.isclose(decision.total_ethical_score, best_score, rtol=1e-9, atol=1e-12)
        ]
        return tied_decisions[0]

    def _create_default_context(self) -> Dict[str, Any]:
        """Provide the budget and route demands used by the example."""
        return {
            "demands": np.array([route.demand_units for route in self.routes]),
            "budget_limit": self.constitutional_ai.ethical_constraints["budget_limit"],
        }

    def _generate_candidates(self, context: Dict[str, Any]) -> List[np.ndarray]:
        """Create representative allocations and rounded random variations."""
        base_allocations = [
            np.array([[3, 2], [1, 2], [0, 1]], dtype=int),
            np.array([[4, 3], [0, 1], [0, 0]], dtype=int),
            np.array([[2, 2], [2, 2], [1, 1]], dtype=int),
            np.array([[2, 2], [1, 2], [1, 1]], dtype=int),
            np.array([[2, 2], [1, 2], [0, 1]], dtype=int),
        ]

        candidates = []
        for base in base_allocations:
            candidates.append(base.copy())
            for _ in range(3):
                variation = np.rint(
                    base + np.random.normal(0, 0.3, base.shape)
                ).astype(int)
                candidates.append(np.maximum(variation, 0))
        return candidates

    def _calculate_objectives(self, fleet_allocation: np.ndarray) -> Dict[str, float]:
        """Calculate four normalized scores, with higher values always preferred."""
        framework = self.constitutional_ai
        total_cost = framework._calculate_total_cost(fleet_allocation)
        baseline = framework.ethical_constraints["baseline_cost"]
        economic = max(0.0, min(1.0, 1.0 - max(0.0, total_cost - baseline) / baseline))

        total_emissions = framework._calculate_total_emissions(fleet_allocation)
        environmental = max(0.0, 1.0 - total_emissions / 10000.0)

        service_levels = framework._service_levels(fleet_allocation)
        social = min(service_levels)
        safety = framework._average_safety(fleet_allocation)

        return {
            "economic": float(economic),
            "environmental": float(environmental),
            "social": float(social),
            "safety": float(safety),
        }

    def _calculate_stakeholder_impacts(
        self, fleet_allocation: np.ndarray
    ) -> Dict[int, Dict[str, float]]:
        """Build bounded proxy impacts from service, cost, and emissions."""
        framework = self.constitutional_ai
        service = float(np.mean(framework._service_levels(fleet_allocation)))
        cost_score = max(
            0.0,
            min(1.0, 1.0 - framework._calculate_total_cost(fleet_allocation) / 500000.0),
        )
        environmental_score = max(
            0.0,
            1.0 - framework._calculate_total_emissions(fleet_allocation) / 10000.0,
        )
        impacts = {}

        for stakeholder in framework.stakeholders:
            # A shared proxy profile makes the role of vulnerability explicit without
            # claiming to measure all effects experienced by each group.
            overall = 0.4 * service + 0.3 * cost_score + 0.3 * environmental_score
            impacts[stakeholder.id] = {
                "service": service,
                "cost": cost_score,
                "environmental": environmental_score,
                "overall": float(overall),
            }
        return impacts

    def _calculate_equity_score(
        self, stakeholder_impacts: Dict[int, Dict[str, float]]
    ) -> float:
        """Measure dispersion in vulnerability-weighted stakeholder proxy impacts."""
        if not stakeholder_impacts:
            return 1.0

        weighted_impacts = []
        stakeholders_by_id = {
            stakeholder.id: stakeholder
            for stakeholder in self.constitutional_ai.stakeholders
        }
        multiplier = self.constitutional_ai.equity_weights["vulnerability_multiplier"]
        for stakeholder_id, impacts in stakeholder_impacts.items():
            stakeholder = stakeholders_by_id[stakeholder_id]
            vulnerability_weight = 1.0 + stakeholder.vulnerability * (multiplier - 1.0)
            weighted_impacts.append(impacts["overall"] * vulnerability_weight)

        mean_impact = float(np.mean(weighted_impacts))
        if np.isclose(mean_impact, 0.0):
            return 1.0 if np.isclose(np.std(weighted_impacts), 0.0) else 0.0
        score = 1.0 - float(np.std(weighted_impacts)) / abs(mean_impact)
        return float(max(0.0, min(1.0, score)))

    def _calculate_sustainability_score(
        self, fleet_allocation: np.ndarray, objectives: Dict[str, float]
    ) -> float:
        """Combine environmental, social, and fleet-size proxy indicators."""
        total_vehicles = float(np.sum(fleet_allocation))
        efficiency_bonus = min(1.0, 15.0 / total_vehicles) if total_vehicles > 0 else 1.0
        return float(
            0.4 * objectives["environmental"]
            + 0.4 * objectives["social"]
            + 0.2 * efficiency_bonus
        )

    def _calculate_transparency_score(self, fleet_allocation: np.ndarray) -> float:
        """Score the example's simple complexity-based reporting proxy."""
        transparency = 0.8
        if np.sum(fleet_allocation) > 10:
            transparency *= 0.9
        if len(self.constitutional_ai.stakeholders) > 4:
            transparency *= 0.95
        return float(min(1.0, transparency))

    def _calculate_total_ethical_score(
        self,
        objectives: Dict[str, float],
        ethical_compliance: Dict[str, float],
        equity_score: float,
        sustainability_score: float,
        transparency_score: float,
    ) -> float:
        """Apply the stated weighted-sum ranking rule."""
        objective_score = sum(
            self.objective_weights[key] * objectives[key]
            for key in self.objective_weights
        )
        compliance_score = sum(
            self.constraint_weights[principle.name]
            * ethical_compliance[key]
            for key, principle in zip(
                ethical_compliance,
                self.constitutional_ai.ethical_principles,
            )
        )
        return float(
            0.3 * objective_score
            + 0.3 * compliance_score
            + 0.2 * equity_score
            + 0.1 * sustainability_score
            + 0.1 * transparency_score
        )

    def _generate_ethical_reasoning(
        self,
        fleet_allocation: np.ndarray,
        objectives: Dict[str, float],
        ethical_compliance: Dict[str, float],
        equity_score: float,
    ) -> List[str]:
        """Summarize the indicators used to rank the selected candidate."""
        best_objective = max(objectives, key=objectives.get)
        average_compliance = float(np.mean(list(ethical_compliance.values())))
        return [
            f"The highest individual objective score is {best_objective} ({objectives[best_objective]:.3f}).",
            f"Mean principle-compliance proxy score: {average_compliance:.3f}.",
            f"Vulnerability-weighted stakeholder dispersion score: {equity_score:.3f}.",
            f"The allocation assigns {int(np.sum(fleet_allocation))} vehicles across the modeled routes.",
            "The selected candidate passed the stated budget, emissions, noise, service, and safety checks.",
        ]

    def _evaluate_candidate(
        self, fleet_allocation: np.ndarray, context: Dict[str, Any]
    ) -> EthicalDecision:
        """Calculate all reported indicators for one feasible allocation."""
        objectives = self._calculate_objectives(fleet_allocation)
        compliance = self.constitutional_ai.evaluate_ethical_compliance(fleet_allocation)
        impacts = self._calculate_stakeholder_impacts(fleet_allocation)
        equity = self._calculate_equity_score(impacts)
        sustainability = self._calculate_sustainability_score(fleet_allocation, objectives)
        transparency = self._calculate_transparency_score(fleet_allocation)
        total_score = self._calculate_total_ethical_score(
            objectives, compliance, equity, sustainability, transparency
        )
        reasoning = self._generate_ethical_reasoning(
            fleet_allocation, objectives, compliance, equity
        )

        return EthicalDecision(
            fleet_allocation=fleet_allocation.copy(),
            objectives=objectives,
            ethical_compliance=compliance,
            stakeholder_impacts=impacts,
            equity_score=equity,
            sustainability_score=sustainability,
            transparency_score=transparency,
            total_ethical_score=total_score,
            reasoning=reasoning,
        )


optimizer = MultiObjectiveOptimizer(constitutional_ai)

## Run the candidate evaluation

The context supplies route demand and the budget limit. In this implementation, route demand is already stored in each `Route`; the context's demand array is retained as a readable record of the problem instance. The objective weights and ethical-principle weights are defined in the framework. The displayed result is produced by executing the cells, not assumed in advance.

In [ ]:
def run_ethical_optimization() -> Tuple[EthicalDecision, float, Dict[str, Any]]:
    """Run the finite-candidate evaluation and return its decision and timing."""
    context = {
        "demands": np.array([route.demand_units for route in routes]),
        "budget_limit": constitutional_ai.ethical_constraints["budget_limit"],
    }

    print("VALUE-ALIGNED FLEET CANDIDATE EVALUATION")
    print("=" * 58)
    print(f"Ethical principles: {len(constitutional_ai.ethical_principles)}")
    print(f"Stakeholder groups: {len(constitutional_ai.stakeholders)}")
    print(f"Objectives: {len(constitutional_ai.objectives)}")
    print(f"Budget limit: {context['budget_limit']:,.0f} monetary units")

    start_time = time.perf_counter()
    decision = optimizer.optimize_fleet(context)
    computation_time = time.perf_counter() - start_time

    print("\nCandidate evaluation completed.")
    print(f"Elapsed computation time: {computation_time:.6f} seconds")
    print(f"Weighted ethical score: {decision.total_ethical_score:.3f}")
    print(f"Equity proxy score: {decision.equity_score:.3f}")
    print(f"Sustainability proxy score: {decision.sustainability_score:.3f}")
    print(f"Transparency proxy score: {decision.transparency_score:.3f}")
    return decision, computation_time, context


ethical_decision, computation_time, optimization_context = run_ethical_optimization()

## Build and inspect the result

The report translates positive matrix entries into vehicle-route assignments and displays each objective and compliance indicator. Compliance dictionary keys are matched to the principle categories used by the evaluator. The computed time is the elapsed time for this run and is not a fixed performance guarantee.

In [ ]:
optimal_fleet = {}
for i, vehicle in enumerate(vehicles):
    for j, route in enumerate(routes):
        count = int(ethical_decision.fleet_allocation[i, j])
        if count > 0:
            optimal_fleet[(vehicle.id, route.id)] = count

stakeholder_equity = {
    stakeholder_id: impacts["overall"]
    for stakeholder_id, impacts in ethical_decision.stakeholder_impacts.items()
}

ethical_result = EthicalResult(
    optimal_fleet=optimal_fleet,
    total_cost=constitutional_ai._calculate_total_cost(ethical_decision.fleet_allocation),
    multi_objective_scores=ethical_decision.objectives,
    ethical_compliance=ethical_decision.ethical_compliance,
    stakeholder_equity=stakeholder_equity,
    sustainability_metrics={
        "sustainability_score": ethical_decision.sustainability_score,
        "environmental_score": ethical_decision.objectives["environmental"],
        "social_score": ethical_decision.objectives["social"],
    },
    transparency_metrics={
        "transparency_score": ethical_decision.transparency_score,
    },
    computation_time=computation_time,
    insights=ethical_decision.reasoning,
)

assert constitutional_ai.is_feasible(
    ethical_decision.fleet_allocation, optimization_context
)
assert np.isclose(
    ethical_result.total_cost,
    constitutional_ai._calculate_total_cost(ethical_decision.fleet_allocation),
)
assert all(
    np.isfinite(score)
    for score in [
        ethical_decision.total_ethical_score,
        ethical_decision.equity_score,
        ethical_decision.sustainability_score,
        ethical_decision.transparency_score,
    ]
)

print("\nSELECTED FEASIBLE CANDIDATE")
print("=" * 58)
for (vehicle_id, route_id), count in ethical_result.optimal_fleet.items():
    vehicle = next(item for item in vehicles if item.id == vehicle_id)
    route = next(item for item in routes if item.id == route_id)
    print(f"  {count} x {vehicle.name} -> {route.name}")

print(f"\nTotal modeled cost: {ethical_result.total_cost:,.2f} monetary units")
print(f"Weighted ethical score: {ethical_decision.total_ethical_score:.3f}")
print(f"Equity proxy score: {ethical_decision.equity_score:.3f}")
print(f"Sustainability proxy score: {ethical_decision.sustainability_score:.3f}")
print(f"Transparency proxy score: {ethical_decision.transparency_score:.3f}")

print("\nObjective scores (higher is preferred):")
for key, score in ethical_result.multi_objective_scores.items():
    print(f"  {constitutional_ai.objectives[key]['name']}: {score:.3f}")

principle_by_key = {
    "environmental": "Environmental Responsibility",
    "social_equity": "Social Equity",
    "safety": "Safety and Well-being",
    "economic": "Economic Viability",
    "transparency": "Transparency and Accountability",
}
print("\nPrinciple-compliance proxy scores:")
for key, score in ethical_result.ethical_compliance.items():
    print(f"  {principle_by_key[key]}: {score:.3f}")

print("\nStakeholder impact proxies:")
for stakeholder_id, impact in ethical_result.stakeholder_equity.items():
    stakeholder = next(item for item in constitutional_ai.stakeholders if item.id == stakeholder_id)
    print(f"  {stakeholder.name}: {impact:.3f}")

print("\nDecision reasoning:")
for item in ethical_result.insights:
    print(f"  - {item}")

## Summary and limitations

The notebook has represented value choices through named principles, stakeholder profiles, hard feasibility checks, and weighted proxy scores. Candidate allocations are generated, filtered, scored, and ranked in that order. The reported allocation is the highest-scoring feasible member of the generated candidate set; random variations mean that the set and the selected result can differ across executions.

The measures are intentionally limited. In particular, the stakeholder impact proxy gives every group the same underlying service, cost, and emissions profile, while vulnerability changes its weight in the equity calculation. It does not model group-specific consequences from the detailed priority dictionaries. The transparency score is a simple complexity-based proxy, not an audit of explainability. Sustainability targets and several route and vehicle attributes are recorded but are not decision variables or direct terms in the scoring rule. These limits should be considered when interpreting the numerical indicators.

### Key takeaways

- A weighted sum makes trade-offs explicit, but its ranking depends on the chosen weights and normalization.
- Hard feasibility checks and soft compliance scores serve different purposes and should be reported separately.
- A finite candidate search can identify the best option in its candidate pool, but cannot establish a global optimum outside that pool.
- Ethical indicators are only as informative as their definitions, evidence, and stakeholder input; aggregate scores do not replace deliberation or accountability.

In [ ]:
print("TIER 8 SUMMARY: VALUE-ALIGNED FLEET EVALUATION")
print("=" * 58)
print("Framework components:")
print(f"  Ethical principles: {len(constitutional_ai.ethical_principles)}")
print(f"  Stakeholder profiles: {len(constitutional_ai.stakeholders)}")
print(f"  Weighted objectives: {len(constitutional_ai.objectives)}")
print("  Hard checks: budget, emissions, noise, service, and safety")
print("  Ranking: weighted score over generated feasible candidates")

print("\nSelected-candidate indicators:")
print(f"  Weighted ethical score: {ethical_decision.total_ethical_score:.3f}")
print(
    "  Mean principle-compliance proxy: "
    f"{np.mean(list(ethical_result.ethical_compliance.values())):.3f}"
)
print(f"  Equity proxy: {ethical_decision.equity_score:.3f}")
print(f"  Sustainability proxy: {ethical_decision.sustainability_score:.3f}")
print(f"  Transparency proxy: {ethical_decision.transparency_score:.3f}")
print(f"  Elapsed evaluation time: {ethical_result.computation_time:.6f} seconds")

print("\nInterpretation:")
print("  The selected allocation is best-scoring among feasible candidates generated in this run.")
print("  The proxy indicators support structured comparison; they are not comprehensive ethical judgments.")